Testing connection 

In [1]:
from pymongo import MongoClient

def connect_to_database():
    try:
        client = MongoClient("mongodb://db:27017/", serverSelectionTimeoutMS=5000)
        db = client.fingerprintDB
        # Teste die Verbindung
        db.command("ping")
        print("Connected to the database.")
        return db
    except Exception as e:
        print(f"Error connecting to MongoDB server: {e}")
        return None

db = connect_to_database()

Connected to the database.


In [1]:
import pymongo
import pandas as pd
import matplotlib.pyplot as plt
import base64
from PIL import Image
from io import BytesIO
import os

# Funktion zur Herstellung der Verbindung zur MongoDB-Datenbank
def connect_to_database():
    try:
        mongo_uri = os.getenv('MONGO_URI', 'mongodb://db:27017/fingerprintDB')
        client = pymongo.MongoClient(mongo_uri)
        db = client.get_default_database()
        print("Connected to the database.")
        return db
    except pymongo.errors.ConnectionError as err:
        print(f"Error: {err}")
        return None

# Funktion zum Abrufen der Benutzer aus der Datenbank
def fetch_users_from_database(db):
    try:
        print("Fetching users from the database...")
        filter = {}
        project = {
            'username': 1,
            '_id': 0
        }
        users = db.fingerprints.find(filter=filter, projection=project)
        user_list = [user["username"] for user in users]
        print(f"Retrieved {len(user_list)} users: {user_list}")
        return user_list
    except Exception as e:
        print(f"Error fetching users: {e}")
        return []

# Funktion zum Abrufen von Fingerabdrücken für einen Benutzer
def fetch_fingerprints_for_user(db, username):
    try:
        print(f"Fetching fingerprints for user: {username}")
        user = db.fingerprints.find_one({"username": username}, {"_id": 1})
        if not user:
            print(f"No user found with username: {username}")
            return []
        
        fingerprint_id = user.get("_id")
        print(f"Retrieved fingerprint ID for user: {username}: {fingerprint_id}")
        return fingerprint_id
    except Exception as e:
        print(f"Error fetching fingerprint ID: {e}")

# Beispiel für die Verwendung der Funktionen
if __name__ == "__main__":    
    db = connect_to_database()
    if db is not None:
        users = fetch_users_from_database(db)
        if users:
            for user in users:
                fetch_fingerprints_for_user(db, user)

Connected to the database.
Fetching users from the database...
Retrieved 0 users: []


In [33]:
import os
import pymongo
import pandas as pd
import matplotlib.pyplot as plt
import base64
from PIL import Image
from io import BytesIO

# Funktion zur Herstellung der Verbindung zur MongoDB-Datenbank
def connect_to_database():
    try:
        mongo_uri = os.getenv('MONGO_URI', 'mongodb://db:27017/fingerprintDB')
        client = pymongo.MongoClient(mongo_uri)
        db = client.get_default_database()
        print("Connected to the database.")
        return db
    except pymongo.errors.ConnectionError as err:
        print(f"Error: {err}")
        return None

# Verbindung zur Datenbank herstellen
db = connect_to_database()
canvassamples_collection = db['canvassamples']
fingerprints_collection = db['fingerprints']

Connected to the database.


1. Datenbereinigung
1.1 Fehlende Werte überprüfen

In [28]:
# Dokumente mit fehlendem 'fingerprintId'-Feld finden
fehlende_fingerprintId = canvassamples_collection.find({ "fingerprintId": { "$exists": False } })
print(f"Anzahl der Dokumente mit fehlendem 'fingerprintId'-Feld: {canvassamples_collection.count_documents({ 'fingerprintId': { '$exists': False } })}")

# Dokumente mit fehlendem 'sampleData'-Feld finden
fehlende_sampleData = canvassamples_collection.find({ "sampleData": { "$exists": False } })
print(f"Anzahl der Dokumente mit fehlendem 'sampleData'-Feld: {canvassamples_collection.count_documents({ 'sampleData': { '$exists': False } })}")

Anzahl der Dokumente mit fehlendem 'fingerprintId'-Feld: 0
Anzahl der Dokumente mit fehlendem 'sampleData'-Feld: 0


1.2 Duplikate entfernen

In [29]:
# Duplikate basierend auf 'fingerprintId' und 'sampleData' entfernen
pipeline = [
    {"$group": {"_id": {"fingerprintId": "$fingerprintId", "sampleData": "$sampleData"}, "count": {"$sum": 1}, "docs": {"$push": "$_id"}}},
    {"$match": {"count": {"$gt": 1}}}
]

duplicates = list(canvassamples_collection.aggregate(pipeline))
for duplicate in duplicates:
    ids_to_remove = duplicate['docs'][1:]  # Behalte das erste Dokument, entferne die restlichen
    canvassamples_collection.delete_many({"_id": {"$in": ids_to_remove}})
print(f"Anzahl der entfernten Duplikate basierend auf 'fingerprintId' und 'sampleData': {len(duplicates)}")

Anzahl der entfernten Duplikate basierend auf 'fingerprintId' und 'sampleData': 0


1.3 Inkonsistente Daten korrigieren


In [30]:
# Inkonsistente Daten in der 'canvassamples'-Collection korrigieren
def validate_and_correct_canvassamples():
    cursor = canvassamples_collection.find()
    for document in cursor:
        update_needed = False
        update_fields = {}

        # Beispielhafte Validierungsregeln
        if 'sampleData' in document and not isinstance(document['sampleData'], str):
            update_fields['sampleData'] = str(document['sampleData'])
            update_needed = True

        if update_needed:
            canvassamples_collection.update_one({'_id': document['_id']}, {'$set': update_fields})
            print(f"Dokument mit ID {document['_id']} in 'canvassamples' wurde aktualisiert.")

validate_and_correct_canvassamples()

# Inkonsistente Daten in der 'fingerprints'-Collection korrigieren
def validate_and_correct_fingerprints():
    cursor = fingerprints_collection.find()
    for document in cursor:
        update_needed = False
        update_fields = {}

        # Beispielhafte Validierungsregeln
        if 'deviceName' in document and not isinstance(document['deviceName'], str):
            update_fields['deviceName'] = str(document['deviceName'])
            update_needed = True

        if 'operatingSystem' in document and not isinstance(document['operatingSystem'], str):
            update_fields['operatingSystem'] = str(document['operatingSystem'])
            update_needed = True

        if 'browser' in document and not isinstance(document['browser'], str):
            update_fields['browser'] = str(document['browser'])
            update_needed = True

        if update_needed:
            fingerprints_collection.update_one({'_id': document['_id']}, {'$set': update_fields})
            print(f"Dokument mit ID {document['_id']} in 'fingerprints' wurde aktualisiert.")

validate_and_correct_fingerprints()

Daten normalisieren und standardisieren

In [34]:
# Beispielhafte Normalisierung und Standardisierung von Daten in der 'canvassamples'-Collection
def normalize_and_standardize_canvassamples():
    cursor = canvassamples_collection.find()
    for document in cursor:
        update_needed = False
        update_fields = {}

        # Normalisierung von 'sampleData'
        if 'sampleData' in document:
            normalized_sample_data = document['sampleData'].strip().lower()
            if normalized_sample_data != document['sampleData']:
                update_fields['sampleData'] = normalized_sample_data
                update_needed = True

        if update_needed:
            canvassamples_collection.update_one({'_id': document['_id']}, {'$set': update_fields})
            print(f"Dokument mit ID {document['_id']} in 'canvassamples' wurde normalisiert und standardisiert.")

normalize_and_standardize_canvassamples()

Daten verknüpfen und Modell trainieren

In [39]:
import os
import pymongo
import pandas as pd
import time

# Funktion zur Herstellung der Verbindung zur MongoDB-Datenbank
def connect_to_database():
    try:
        mongo_uri = os.getenv('MONGO_URI', 'mongodb://db:27017/fingerprintDB')
        client = pymongo.MongoClient(mongo_uri)
        db = client.get_default_database()
        print("Connected to the database.")
        return db
    except pymongo.errors.ConnectionError as err:
        print(f"Error: {err}")
        return None

# Verbindung zur Datenbank herstellen
db = connect_to_database()
canvassamples_collection = db['canvassamples']
fingerprints_collection = db['fingerprints']

# Daten aus beiden Collections verknüpfen
def get_combined_data():
    start_time = time.time()
    combined_data = []

    # Alle Fingerprints abrufen
    fingerprints_cursor = fingerprints_collection.find()
    for fingerprint in fingerprints_cursor:
        username = fingerprint["username"]
        fingerprint_id = fingerprint["_id"]

        # Alle SampleData für den aktuellen Fingerprint abrufen
        samples_cursor = canvassamples_collection.find({"fingerprintId": fingerprint_id})
        for sample in samples_cursor:
            combined_entry = {
                "sampleData": sample["sampleData"],
                "username": username
            }
            combined_data.append(combined_entry)

    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Data loaded in {elapsed_time:.2f} seconds.")
    return pd.DataFrame(combined_data)

# Kombinierte Daten abrufen
combined_data = get_combined_data()
print(combined_data.head())

# Anzahl der geladenen Daten anzeigen
print(f"Total number of records: {len(combined_data)}")

# Beispielhaft drei SampleData-Einträge pro Benutzername anzeigen
sample_data_per_user = combined_data.groupby('username').head(3)
print(sample_data_per_user)

Connected to the database.
Data loaded in 0.16 seconds.
                                          sampleData        username
0  ivborw0kggoaaaansuheugaaargaaaajcayaaabprbvwaa...  benutzername_1
1  ivborw0kggoaaaansuheugaaargaaaajcayaaabprbvwaa...  benutzername_3
2  ivborw0kggoaaaansuheugaaargaaaajcayaaabprbvwaa...  benutzername_7
3  ivborw0kggoaaaansuheugaaargaaaajcayaaabprbvwaa...  benutzername_4
4  ivborw0kggoaaaansuheugaaargaaaajcayaaabprbvwaa...  benutzername_5
Total number of records: 20
                                           sampleData         username
0   ivborw0kggoaaaansuheugaaargaaaajcayaaabprbvwaa...   benutzername_1
1   ivborw0kggoaaaansuheugaaargaaaajcayaaabprbvwaa...   benutzername_3
2   ivborw0kggoaaaansuheugaaargaaaajcayaaabprbvwaa...   benutzername_7
3   ivborw0kggoaaaansuheugaaargaaaajcayaaabprbvwaa...   benutzername_4
4   ivborw0kggoaaaansuheugaaargaaaajcayaaabprbvwaa...   benutzername_5
5   ivborw0kggoaaaansuheugaaargaaaajcayaaabprbvwaa...  benutzername_10
6   i